In [1]:
import torch
import torch.nn.functional as F

In [ ]:
words = open('names.txt').read().splitlines()
words[:3]

In [ ]:
unique = sorted(list(set(''.join(words))))
unique[:3]

In [ ]:
c_to_idx = {}
idx_to_c = {}

for idx, c in enumerate(unique):
    c_to_idx[c] = idx + 1
    idx_to_c[idx + 1] = c

c_to_idx['.'] = 0
idx_to_c[0] = '.'

In [ ]:
print(c_to_idx)

In [ ]:
print(idx_to_c)

In [ ]:
x = []
y = []

for word in words[:3]:
    sep_word = list('.') + list(word) + list('.')

    for ch1, ch2 in zip(sep_word, sep_word[1:]):
        idx1 = c_to_idx[ch1]
        idx2 = c_to_idx[ch2]

        x.append(idx1)
        y.append(idx2)

x = torch.tensor(x)
y = torch.tensor(y)

print(x)
print(y)

In [ ]:
# input matrix of size (b, 27)

enc_x = F.one_hot(x, num_classes=27).float()
print(enc_x.dtype)
enc_x.shape

In [ ]:
# weight matrix 

w = torch.randn((27, 27), requires_grad=True) # gives a weight matrix initialized in the normal distribution

print(w.dtype)
w.shape

In [ ]:
logits = enc_x @ w
# (10, 27) * (27, 27) => (10, 27)
logits.shape

In [ ]:
exp = logits.exp()
print(exp.shape)
exp

In [ ]:
test = exp.sum(dim=1)
print(test.shape)
test # tensor with sum of each row (each input) that is given to the neural network

In [ ]:
counts = exp.sum(dim=1, keepdim=True)
print(counts.shape)
counts

In [ ]:
probs = exp / counts # this is brodcastible and hence works
probs

In [ ]:
sum(probs[1])

In [ ]:
# calculating loss

nll = torch.zeros(len(y)) # stores the negative log loss for each input
print(len(y))
for i in range(len(y)): # for each output we will get a loss

    prob = probs[i, y[i]]
    log_loss = torch.log(prob)
    neg_log_loss = -log_loss
    nll[i] = neg_log_loss

loss = nll.mean()
loss

In [ ]:
# backward prop

w.grad = None # set gradients to zero
w.grad

loss.backward()

In [ ]:
w.data += -0.1 * w.grad

In [2]:
# ok now setting all of this up in clean cells 

words = open('names.txt').read().splitlines()

unique = sorted(list(set(''.join(words))))

c_to_idx = {}
idx_to_c = {}

for idx, c in enumerate(unique):
    c_to_idx[c] = idx + 1
    idx_to_c[idx + 1] = c

c_to_idx['.'] = 0
idx_to_c[0] = '.'

x = []
y = []

for word in words:
    sep_word = list('.') + list(word) + list('.')

    for ch1, ch2 in zip(sep_word, sep_word[1:]):
        idx1 = c_to_idx[ch1]
        idx2 = c_to_idx[ch2]

        x.append(idx1)
        y.append(idx2)

x = torch.tensor(x)
y = torch.tensor(y)

print(x)
print(y)

tensor([ 0,  5, 13,  ..., 25, 26, 24])
tensor([ 5, 13, 13,  ..., 26, 24,  0])


In [3]:
# input matrix of size (b, 27)

enc_x = F.one_hot(x, num_classes=27).float()
print(enc_x.dtype)

# generator
g = torch.Generator().manual_seed(2147483647)

# weight matrix 
w = torch.randn((27, 27), generator=g, requires_grad=True) # gives a weight matrix initialized in the normal distribution


torch.float32


In [8]:
# gradient descent

epochs = 100
lr = 50

for i in range(epochs):

    # forward prop

    logits = enc_x @ w
    # (10, 27) * (27, 27) => (10, 27)
    
    # softmax
    exp = logits.exp()
    counts = exp.sum(dim=1, keepdim=True)
    probs = exp / counts # this is brodcastible and hence works

    # calculating loss
    nll = torch.zeros(len(y)) # stores the negative log loss for each input

    '''
    for i in range(len(y)): # for each output we will get a loss

        prob = probs[i, y[i]]
        log_loss = torch.log(prob)
        neg_log_loss = -log_loss
        nll[i] = neg_log_loss

    loss = nll.mean()
    very slow and can be done using one vectorized PyTorch operation
    '''
    
    loss = -probs[torch.arange(len(y)), y].log().mean()

    print(loss.item())

    # backward prop
    w.grad = None # setting gradients to zero
    loss.backward()

    w.data += -lr * w.grad
    
    

2.4623453617095947
2.462298631668091
2.462252140045166
2.4622061252593994
2.462161064147949
2.46211576461792
2.462071657180786
2.4620275497436523
2.461984395980835
2.4619412422180176
2.4618983268737793
2.4618561267852783
2.4618144035339355
2.461773157119751
2.4617323875427246
2.4616920948028564
2.4616520404815674
2.4616122245788574
2.4615728855133057
2.461534023284912
2.4614953994750977
2.4614572525024414
2.4614193439483643
2.4613819122314453
2.4613449573516846
2.461308240890503
2.4612717628479004
2.461236000061035
2.46120023727417
2.461164712905884
2.461130142211914
2.461095094680786
2.4610607624053955
2.461026906967163
2.4609932899475098
2.4609594345092773
2.4609262943267822
2.4608936309814453
2.4608609676361084
2.4608285427093506
2.460797071456909
2.4607653617858887
2.460733652114868
2.460702419281006
2.460671901702881
2.460641384124756
2.460610866546631
2.460580587387085
2.4605510234832764
2.4605214595794678
2.4604921340942383
2.460462808609009
2.4604339599609375
2.4604055881500244

In [9]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ w # predict log-counts
        counts = logits.exp() # equivalent to N matrix
        p = counts / counts.sum(1, keepdim=True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(idx_to_c[ix])
        if ix == 0:
            break
    print(''.join(out))

junide.
janasah.
prelay.
a.
nn.
